# DocuMind AI
### Multi-Document Retrieval-Augmented Generation with Source Citations
**Topic 183 — Generative AI Model Making Project**

This notebook builds the complete pipeline end-to-end, in the same order as the project's
implementation flow:

`Ingest documents → Chunk → Embed → Vector database → Retrieve → LLM generation → Citations → Evaluation → Gradio demo`

Run the cells **top to bottom**. The first run downloads two small open models
(an embedding model and a generation model) from Hugging Face — this needs
internet access, which a normal Colab runtime has.

> Swap in your own PDFs any time by uploading them in the **Step 1** cell instead of using the
> bundled sample "College Academic Assistant" documents.


## Setup — install dependencies

In [ ]:
!pip -q install pypdf sentence-transformers faiss-cpu transformers accelerate gradio reportlab
print("done")


done


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

## Step 1 — Document Ingestion
We need a handful of PDFs to demonstrate the system. Run the cell below to generate five
realistic sample academic documents (a GenAI implementation guide, academic rules, project
guidelines, placement guidelines and a student handbook) — this is the "College Academic
Assistant" theme. If you'd rather use your **own** PDFs, run the *optional upload cell* right
after it instead.

In [ ]:
import os
from reportlab.lib.pagesizes import LETTER
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

DOCS_DIR = "documents"
os.makedirs(DOCS_DIR, exist_ok=True)

styles = getSampleStyleSheet()
H1 = ParagraphStyle("H1", parent=styles["Heading1"], spaceAfter=12)
H2 = ParagraphStyle("H2", parent=styles["Heading2"], spaceAfter=8, spaceBefore=14)
BODY = ParagraphStyle("Body", parent=styles["BodyText"], spaceAfter=10, leading=15)

def write_pdf(filename, title, sections):
    path = os.path.join(DOCS_DIR, filename)
    doc = SimpleDocTemplate(path, pagesize=LETTER, topMargin=0.9*inch, bottomMargin=0.9*inch,
                             leftMargin=0.9*inch, rightMargin=0.9*inch)
    story = [Paragraph(title, H1), Spacer(1, 0.15*inch)]
    for heading, paras in sections:
        story.append(Paragraph(heading, H2))
        for p in paras:
            story.append(Paragraph(p, BODY))
    doc.build(story)
    print("wrote", path)

write_pdf("GenAI_Implementation_Guide.pdf", "Generative AI \u2014 Model Making: Implementation Guide", [
    ("Overview of the Model Making Track", [
        "The Model Making track requires every group to design, build and demonstrate a working "
        "Generative AI system across two internal juries (Jury 1 and Jury 2) followed by an End "
        "Semester Examination (ESE) jury. The same project topic is carried through all three "
        "evaluation stages so that the work can be progressively deepened rather than restarted."]),
    ("Jury 1 Requirements", [
        "Jury 1 is the first presentation of the selected topic. Groups must submit a problem "
        "statement, a proposed Generative AI solution, the intended architecture, and a basic "
        "working prototype covering at least one end-to-end pass of the pipeline (for example: "
        "input, processing and a generated output). A live demonstration of the prototype is "
        "mandatory for Jury 1."]),
    ("Jury 2 Requirements", [
        "Model Making groups continue with the same topic for Jury 2 and extend and deepen the "
        "model developed for Jury 1. Jury 2 expects improved accuracy or robustness, additional "
        "features identified as future work in Jury 1, and a more complete evaluation of the "
        "system on a larger set of test cases. Changing the core topic between Jury 1 and Jury 2 "
        "is not permitted."]),
    ("Can the Topic Change After Jury 1?", [
        "No. For the Model Making category, the topic finalized at Jury 1 must be carried forward "
        "to Jury 2 and the ESE jury. Only the depth, features and evaluation rigor are expected to "
        "increase; the underlying problem and Generative AI approach should remain the same."]),
    ("ESE Jury Requirements", [
        "The End Semester Examination (ESE) jury requires substantial and in-depth work on the "
        "selected topic, going beyond what was shown at Jury 2. Groups are expected to present a "
        "polished, interactive application, a complete evaluation covering accuracy and reliability "
        "metrics, and a clear articulation of the underlying Generative AI concepts to the panel. "
        "Every group member must be able to answer questions about the section of the project they "
        "were responsible for."]),
    ("Recommended Technology Stack", [
        "For Retrieval-Augmented Generation style Model Making projects, the recommended stack is "
        "Python developed in Google Colab, PyPDF for document parsing, Sentence Transformers for "
        "embeddings, FAISS as the vector database, an open or API-based Large Language Model for "
        "generation, and Gradio for the final interactive interface."]),
    ("Group Work Division", [
        "Groups of four are expected to divide responsibilities across document processing, "
        "embeddings and retrieval, LLM and prompt engineering, and interface plus evaluation, while "
        "every member retains a working understanding of the complete pipeline."]),
])

write_pdf("Academic_Rules_and_Regulations.pdf", "Academic Rules and Regulations", [
    ("Attendance Requirement", [
        "A minimum of 75% attendance is required in each course, including lectures, practicals and "
        "tutorials, for a student to be eligible to appear for the End Semester Examination. "
        "Students with attendance between 65% and 75% may be granted condonation only with valid "
        "medical or documented grounds, subject to approval by the department committee."]),
    ("Examination Eligibility", [
        "Students must clear all internal assessments with a minimum aggregate of 40% to be "
        "eligible for the End Semester Examination. Backlogs from previous semesters must be "
        "cleared as per the academic calendar published by the examination cell."]),
    ("Grading Policy", [
        "The university follows a ten-point grading scale. Final grades are computed from a "
        "weighted combination of internal assessment, mid-semester evaluation and the End Semester "
        "Examination, with weightages published separately for theory and laboratory courses."]),
    ("Malpractice and Discipline", [
        "Any form of academic malpractice during internal assessments, project evaluations or "
        "examinations, including plagiarism in project reports, will be referred to the Unfair "
        "Means Committee and may result in penalties ranging from a warning to cancellation of the "
        "examination."]),
])

write_pdf("Project_Guidelines.pdf", "Final Year Project and Model Making Guidelines", [
    ("Project Categories", [
        "Final-year Generative AI projects may be undertaken under one of three categories: Model "
        "Making, Real-Time Scenario, or Case Study. Each category has a distinct evaluation rubric "
        "and distinct expectations for continuity of topic across juries."]),
    ("Model Making Category", [
        "In the Model Making category, groups build and demonstrate an actual working Generative AI "
        "system. The same topic and system are carried through Jury 1, Jury 2 and the ESE jury, "
        "with each stage expected to add depth, robustness or new capabilities to the same "
        "underlying model."]),
    ("Real-Time Scenario Category", [
        "Groups in the Real-Time Scenario category continue with the same topic across juries as "
        "well, applying their Generative AI solution to a live or simulated real-world scenario and "
        "demonstrating its behaviour under realistic conditions."]),
    ("Case Study Category", [
        "The Case Study category requires groups to select a new topic for the ESE stage, focusing "
        "on an in-depth analysis and comparison of existing Generative AI approaches rather than "
        "continuing a single built system."]),
    ("Group Size and Formation", [
        "Projects are undertaken in groups of four students. Group formation must be finalized "
        "before the topic proposal is submitted, and changes to group composition after Jury 1 "
        "require written approval from the project coordinator."]),
    ("Report Submission", [
        "Each group must submit a project report following the department template after every "
        "jury, documenting the problem statement, methodology, architecture, implementation and "
        "evaluation results relevant to that stage."]),
])

write_pdf("Placement_Guidelines.pdf", "Placement Cell \u2014 Guidelines for Final Year Students", [
    ("Eligibility Criteria", [
        "Students must have a minimum aggregate CGPA of 6.0 with no active backlogs at the time of "
        "registration to be eligible for on-campus placement drives. Some recruiters may specify a "
        "higher CGPA cutoff, which will be published along with the job description."]),
    ("Registration Process", [
        "Students must register on the placement portal at the start of the final year and keep "
        "their resume and academic details updated before each placement season begins."]),
    ("Placement Rounds", [
        "A typical placement process includes an online aptitude test, a technical interview, and a "
        "human resources interview. Some companies additionally conduct a group discussion or a "
        "coding round before shortlisting candidates for interviews."]),
    ("Offer Policy", [
        "Students who accept a placement offer through the campus process are expected to honour "
        "that commitment and are generally not permitted to sit for further placement drives in the "
        "same academic year, except for a higher package dream-company drive as defined by "
        "placement policy."]),
])

write_pdf("Student_Handbook.pdf", "Student Handbook", [
    ("Code of Conduct", [
        "Students are expected to maintain discipline and integrity on campus, including during "
        "project work, laboratory sessions and examinations. Ragging in any form is strictly "
        "prohibited and punishable under university and legal provisions."]),
    ("Library Facilities", [
        "The central library provides access to technical journals, e-books and a dedicated "
        "Generative AI and Data Science reading section. Final-year project groups may reserve "
        "discussion rooms for project meetings during working hours."]),
    ("Grievance Redressal", [
        "Students may raise academic or administrative grievances, including disputes related to "
        "project evaluation, through the departmental grievance redressal committee within two "
        "weeks of the concerned event."]),
    ("Extracurricular Participation", [
        "Students are encouraged to participate in technical fests, hackathons and Generative AI "
        "workshops organized by the department, subject to maintaining the minimum attendance "
        "requirement in regular coursework."]),
])

print("\nSample documents ready in ./documents")


ModuleNotFoundError: No module named 'reportlab'

**Optional — use your own PDFs instead:** run this cell and select files; it will *replace*
the sample set with whatever you upload (skip it to keep the bundled sample documents).

In [ ]:
# Optional: uncomment to upload your own PDFs in place of the sample set
# from google.colab import files
# import shutil
# uploaded = files.upload()
# shutil.rmtree(DOCS_DIR, ignore_errors=True)
# os.makedirs(DOCS_DIR, exist_ok=True)
# for name in uploaded:
#     shutil.move(name, os.path.join(DOCS_DIR, name))
# print(sorted(os.listdir(DOCS_DIR)))


## Step 2 — Text Extraction
Read every PDF page-by-page with **PyPDF**, keeping the document name and page number attached
to every page. This is the metadata that makes citations possible later.

In [ ]:
from pypdf import PdfReader
from dataclasses import dataclass

@dataclass
class Page:
    doc_name: str
    page_num: int
    text: str

def load_pdfs(folder):
    pages = []
    for fname in sorted(f for f in os.listdir(folder) if f.lower().endswith(".pdf")):
        reader = PdfReader(os.path.join(folder, fname))
        for i, page in enumerate(reader.pages):
            pages.append(Page(doc_name=fname, page_num=i + 1, text=page.extract_text() or ""))
    return pages

pages = load_pdfs(DOCS_DIR)
print(f"Extracted {len(pages)} pages from {len({p.doc_name for p in pages})} document(s)")
print("\n--- sample page ---")
print(pages[0].doc_name, "| page", pages[0].page_num)
print(pages[0].text[:400])


## Step 3 — Text Cleaning
Light-touch cleanup: re-join words that were hyphen-broken across a line, collapse extra
whitespace, and drop stray page-number-only lines — while keeping the document's actual wording
intact, since that wording is what the LLM will quote from later.

In [ ]:
import re

_WS = re.compile(r"[ \t]+")
_BLANK = re.compile(r"\n{3,}")
_HYPHEN = re.compile(r"(\w)-\n(\w)")

def clean_text(text):
    text = _HYPHEN.sub(r"\1\2", text)
    text = _WS.sub(" ", text)
    text = _BLANK.sub("\n\n", text)
    lines = [ln for ln in text.split("\n") if ln.strip() and not ln.strip().isdigit()]
    return "\n".join(lines).strip()

for p in pages:
    p.text = clean_text(p.text)
pages = [p for p in pages if p.text]
print("Pages remaining after cleaning (blank/scanned pages dropped):", len(pages))


## Step 4 — Chunking
We don't hand an entire document to the LLM — large inputs are expensive and make it harder for
the model to focus on what matters. Instead we split each page into smaller, overlapping chunks
(default: 800 characters, 150 overlap) and keep a `chunk_id`, `doc_name` and `page_num` on every
one — this is exactly what lets us cite an exact source later.

In [ ]:
import itertools
_counter = itertools.count()

@dataclass
class Chunk:
    chunk_id: str
    doc_name: str
    page_num: int
    text: str

def chunk_pages(pages, chunk_size=800, overlap=150):
    chunks = []
    for page in pages:
        text = page.text
        if len(text) <= chunk_size:
            chunks.append(Chunk(f"c{next(_counter):05d}", page.doc_name, page.page_num, text.strip()))
            continue
        start = 0
        while start < len(text):
            end = min(start + chunk_size, len(text))
            piece = text[start:end].strip()
            chunks.append(Chunk(f"c{next(_counter):05d}", page.doc_name, page.page_num, piece))
            if end == len(text):
                break
            start = end - overlap
    return chunks

chunks = chunk_pages(pages)
print(f"Split {len(pages)} pages into {len(chunks)} chunks")
print("\n--- sample chunk ---")
print(chunks[0].chunk_id, "|", chunks[0].doc_name, "| page", chunks[0].page_num)
print(chunks[0].text[:300])


## Step 5 — Embeddings
An embedding model (**Sentence Transformers**, `all-MiniLM-L6-v2`) converts each chunk of text
into a numerical vector that represents its *meaning*. Texts with similar meaning land close
together in vector space — even if they barely share any words. Let's prove that with the exact
example from the project brief before embedding the real documents.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

a, b, c = embedder.encode([
    "Jury 2 requirements",
    "What happens during the second jury?",
    "What is the boiling point of tungsten?",
], normalize_embeddings=True)

def cos(x, y):
    return float(np.dot(x, y))

print("similarity('Jury 2 requirements', 'what happens during the second jury?') =", round(cos(a, b), 3))
print("similarity('Jury 2 requirements', 'boiling point of tungsten')            =", round(cos(a, c), 3))
print("\n-> different wording, same meaning, scores much higher than an unrelated sentence.")


In [ ]:
chunk_vectors = embedder.encode([c.text for c in chunks], normalize_embeddings=True, show_progress_bar=True)
chunk_vectors = np.asarray(chunk_vectors, dtype="float32")
print("Embedded", chunk_vectors.shape[0], "chunks into", chunk_vectors.shape[1], "-dim vectors")


## Step 6 — Vector Database (FAISS)
Store every chunk's embedding in a FAISS index for fast similarity search. We use the inner
product on normalized vectors, which is equivalent to cosine similarity.

In [ ]:
import faiss

index = faiss.IndexFlatIP(chunk_vectors.shape[1])
index.add(chunk_vectors)
print("FAISS index built with", index.ntotal, "vectors")


## Step 7 — Retrieval
Convert the user's question into an embedding with the *same* model, then ask FAISS for the
Top-K most similar chunks.

In [ ]:
def retrieve(question, top_k=4):
    q_vec = embedder.encode([question], normalize_embeddings=True).astype("float32")
    scores, idxs = index.search(q_vec, top_k)
    return [(chunks[i], float(s)) for s, i in zip(scores[0], idxs[0]) if i != -1]

demo_hits = retrieve("What happens in Jury 2 for a Model Making project?")
for chunk, score in demo_hits:
    print(f"[{score:.3f}] {chunk.doc_name} (page {chunk.page_num}): {chunk.text[:100]}...")


## Step 8 — Prompt Construction
This is the key grounding step: the LLM is explicitly instructed to answer **only** from the
retrieved context, and to say so plainly if the context is not enough — not to fall back on its
own pretrained knowledge.

In [ ]:
NOT_FOUND_MESSAGE = "I could not find sufficient information about this in the provided documents."

SYSTEM_INSTRUCTION = (
    "You are an academic assistant. Answer the user's question using ONLY the provided document "
    "context. Do not use any outside knowledge. If the context does not contain enough information "
    f'to answer, respond exactly with: "{NOT_FOUND_MESSAGE}"'
)

def build_prompt(question, retrieved_chunks):
    blocks = [f"[{i+1}] (Source: {c.doc_name}, Page {c.page_num})\n{c.text}"
              for i, c in enumerate(retrieved_chunks)]
    context = "\n\n".join(blocks)
    return (f"{SYSTEM_INSTRUCTION}\n\nContext:\n{context}\n\n"
            f"Question: {question}\n\nAnswer (cite which numbered source you used):")

print(build_prompt("What happens in Jury 2 for a Model Making project?", [h[0] for h in demo_hits]))


## Step 9 — LLM Generation
We use `google/flan-t5-base` — a small, open, instruction-tuned model that needs no API key and
runs comfortably on Colab's CPU runtime. (To use an API-based LLM instead — OpenAI, Gemini,
Claude, etc. — just replace `generate()` below with a call to that API; the rest of the pipeline
does not change.)

In [ ]:
from transformers import pipeline

generator = pipeline("text2text-generation", model="google/flan-t5-base")

def generate(prompt, max_new_tokens=200):
    out = generator(prompt, max_new_tokens=max_new_tokens, do_sample=False)
    return out[0]["generated_text"].strip()

demo_prompt = build_prompt("What happens in Jury 2 for a Model Making project?", [h[0] for h in demo_hits])
print(generate(demo_prompt))


## Step 10 — Citations & Full Pipeline
Now we tie retrieval + prompting + generation + citation together into one `ask()` function, with
a grounding safeguard: if even the *best* retrieved chunk is a weak match (low similarity score),
we refuse to answer rather than let the LLM guess from thin context.

In [ ]:
MIN_SIMILARITY = 0.30  # tune this against your own documents/embedding model

def ask(question, top_k=4):
    hits = retrieve(question, top_k=top_k)
    if not hits or hits[0][1] < MIN_SIMILARITY:
        return {"answer": NOT_FOUND_MESSAGE, "grounded": False, "citations": [], "hits": hits}

    retrieved_chunks = [c for c, _ in hits]
    prompt = build_prompt(question, retrieved_chunks)
    raw_answer = generate(prompt)
    grounded = NOT_FOUND_MESSAGE.lower() not in raw_answer.lower()
    citations = [f"{retrieved_chunks[0].doc_name} \u2014 Page {retrieved_chunks[0].page_num}"] if grounded else []
    return {"answer": raw_answer, "grounded": grounded, "citations": citations, "hits": hits}


def show(question):
    result = ask(question)
    print("Q:", question)
    print("A:", result["answer"])
    if result["citations"]:
        print("\U0001F4C4 Source:", "; ".join(result["citations"]))
    print("-" * 90)

show("What happens in Jury 2 for a Model Making project?")
show("Can we change our Model Making topic after Jury 1?")
show("What is the minimum attendance percentage required to appear for the ESE?")


## Step 11 — Hallucination Control / Grounding Test
The whole point of RAG is that the system should say "I don't know" instead of inventing an
answer when the documents genuinely don't cover something. Let's test that directly.

In [ ]:
show("What is the exact attendance percentage for an event not mentioned in the documents?")
show("What is the boiling point of tungsten on Mars?")
show("What is the WiFi password for the student hostel?")


## Step 12 — Evaluation
A test set of 27 questions — single-document, cross-document, and deliberately unanswerable —
run through the pipeline. `retrieval_correct` and the refusal check are computed automatically;
`answer_correct` / `citation_correct` are left for you to fill in after reading each transcript
against the source PDF (free-text correctness needs a human judge, not a hard-coded rule).

**Note:** these numbers reflect *this* run, with *this* embedding/LLM choice and threshold — they
are not a universal benchmark. Re-run after any change to see the effect.

In [ ]:
import csv

eval_questions = [
    ("1","single-document","What happens in Jury 2 for a Model Making project?","GenAI_Implementation_Guide.pdf"),
    ("2","single-document","Can we change our Model Making topic after Jury 1?","GenAI_Implementation_Guide.pdf"),
    ("3","single-document","What does the ESE jury require?","GenAI_Implementation_Guide.pdf"),
    ("4","single-document","What is the recommended technology stack for RAG projects?","GenAI_Implementation_Guide.pdf"),
    ("5","single-document","How should a group of four divide responsibilities on a Model Making project?","GenAI_Implementation_Guide.pdf"),
    ("6","single-document","What is the minimum attendance percentage required to appear for the ESE?","Academic_Rules_and_Regulations.pdf"),
    ("7","single-document","What CGPA aggregate is required to be eligible for the End Semester Examination?","Academic_Rules_and_Regulations.pdf"),
    ("8","single-document","What grading scale does the university follow?","Academic_Rules_and_Regulations.pdf"),
    ("9","single-document","What happens in case of academic malpractice such as plagiarism in a project report?","Academic_Rules_and_Regulations.pdf"),
    ("10","single-document","What are the three project categories available for the final year project?","Project_Guidelines.pdf"),
    ("11","single-document","Does the topic change between juries in the Case Study category?","Project_Guidelines.pdf"),
    ("12","single-document","What is the group size for final year projects?","Project_Guidelines.pdf"),
    ("13","single-document","What must a project report document after every jury?","Project_Guidelines.pdf"),
    ("14","single-document","What is the minimum CGPA required for on-campus placement eligibility?","Placement_Guidelines.pdf"),
    ("15","single-document","What are the typical rounds in a placement process?","Placement_Guidelines.pdf"),
    ("16","single-document","What happens if a student accepts a placement offer through campus placement?","Placement_Guidelines.pdf"),
    ("17","single-document","What facilities does the central library provide?","Student_Handbook.pdf"),
    ("18","single-document","How should a student raise a grievance about project evaluation?","Student_Handbook.pdf"),
    ("19","single-document","Is ragging allowed on campus?","Student_Handbook.pdf"),
    ("20","cross-document","If a student wants to carry their Model Making topic into the ESE jury and also be allowed to sit the exam, what does the implementation guide say about the topic and what does the academic rules document say about eligibility?","GenAI_Implementation_Guide.pdf;Academic_Rules_and_Regulations.pdf"),
    ("21","cross-document","How does the project-category topic-continuity rule compare with the ESE examination eligibility requirement?","Project_Guidelines.pdf;Academic_Rules_and_Regulations.pdf"),
    ("22","cross-document","What do the placement guidelines say about accepting an offer, and what does the implementation guide say about finishing the Model Making project \u2014 could both be happening at once?","Placement_Guidelines.pdf;GenAI_Implementation_Guide.pdf"),
    ("23","cross-document","What do the Student Handbook and the Project Guidelines each say about spaces or conduct expectations during project work?","Student_Handbook.pdf;Project_Guidelines.pdf"),
    ("24","unanswerable","What is the exact attendance percentage for an event that is not mentioned in any of the documents?","NONE"),
    ("25","unanswerable","What is the fee refund policy for a cancelled semester?","NONE"),
    ("26","unanswerable","Which cloud provider does the placement cell recommend for technical interviews?","NONE"),
    ("27","unanswerable","What is the WiFi password for the student hostel?","NONE"),
]

rows = []
n_answerable = n_retrieval_correct = n_unanswerable = n_refused_correctly = 0

for qid, category, question, expected in eval_questions:
    expected_sources = expected.split(";")
    expect_answerable = expected_sources != ["NONE"]
    result = ask(question)
    top_doc = result["hits"][0][0].doc_name if result["hits"] else None

    retrieval_correct = ""
    if expect_answerable:
        n_answerable += 1
        retrieval_correct = bool(top_doc and top_doc in expected_sources)
        n_retrieval_correct += int(retrieval_correct)
    else:
        n_unanswerable += 1
        n_refused_correctly += int(not result["grounded"])

    rows.append({
        "id": qid, "category": category, "question": question, "expected_source": expected,
        "top_retrieved_doc": top_doc or "", "retrieval_correct(auto)": retrieval_correct,
        "system_answer": result["answer"], "grounded": result["grounded"],
        "citation": "; ".join(result["citations"]),
        "answer_correct(fill in manually)": "", "citation_correct(fill in manually)": "",
    })

with open("eval_results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

print(f"Retrieval accuracy (answerable questions):        {n_retrieval_correct}/{n_answerable} = {n_retrieval_correct/n_answerable:.0%}")
print(f"Correct refusal rate (unanswerable questions):    {n_refused_correctly}/{n_unanswerable} = {n_refused_correctly/n_unanswerable:.0%}")
print("\nFull results written to eval_results.csv \u2014 open it, read each answer against the source PDF,")
print("and fill in answer_correct / citation_correct by hand before reporting final numbers.")


In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 60)
pd.read_csv("eval_results.csv")


## Step 13 — Interactive Demo (Gradio)
The final interface: upload documents (or keep using the bundled sample set), ask a question, see
the grounded answer with its citation. `share=True` gives you a public link you can hand to the
jury during a live demo.

In [ ]:
import gradio as gr
import tempfile, shutil

def gradio_load_documents(files):
    global chunks, chunk_vectors, index
    folder = DOCS_DIR
    if files:
        folder = tempfile.mkdtemp(prefix="documind_upload_")
        for f in files:
            shutil.copy(f.name, os.path.join(folder, os.path.basename(f.name)))

    new_pages = [p for p in (Page(fn, i+1, clean_text(pg.extract_text() or ""))
                              for fn in sorted(x for x in os.listdir(folder) if x.lower().endswith(".pdf"))
                              for i, pg in enumerate(PdfReader(os.path.join(folder, fn)).pages))
                 if p.text]
    chunks = chunk_pages(new_pages)
    chunk_vectors = embedder.encode([c.text for c in chunks], normalize_embeddings=True).astype("float32")
    index = faiss.IndexFlatIP(chunk_vectors.shape[1])
    index.add(chunk_vectors)
    doc_names = sorted({c.doc_name for c in chunks})
    return f"Indexed {len(chunks)} chunks from {len(doc_names)} document(s): " + ", ".join(doc_names)

def gradio_ask(question):
    if not question.strip():
        return "Please enter a question.", ""
    result = ask(question)
    citation_md = "\n".join(f"- {c}" for c in result["citations"]) if result["citations"] else "_(no source \u2014 not found in the documents)_"
    return result["answer"], citation_md

with gr.Blocks(title="DocuMind AI") as demo:
    gr.Markdown("# DocuMind AI\nMulti-Document RAG with Source Citations \u2014 Topic 183")
    files_in = gr.File(label="Upload PDF documents (optional \u2014 leave empty to use the sample set)",
                        file_count="multiple", file_types=[".pdf"])
    process_btn = gr.Button("Process Documents")
    status = gr.Textbox(label="Index status", interactive=False)
    process_btn.click(gradio_load_documents, inputs=[files_in], outputs=[status])

    gr.Markdown("---")
    question_in = gr.Textbox(label="Ask a question", placeholder="e.g. What happens in Jury 2 for a Model Making project?")
    ask_btn = gr.Button("Ask AI")
    answer_out = gr.Textbox(label="Answer", interactive=False)
    citation_out = gr.Markdown(label="Source")
    ask_btn.click(gradio_ask, inputs=[question_in], outputs=[answer_out, citation_out])

demo.launch(share=True, debug=False)


## Wrap-up
You now have every piece the problem statement asks for: ingestion, chunking, embeddings, a
FAISS vector database, Top-K retrieval, an LLM answering strictly from retrieved context,
inline source citations, a working refusal path for out-of-scope questions, an evaluation
harness across single-document / cross-document / unanswerable questions, and a live Gradio demo.

**Next steps for Jury 2 / ESE**, in the same spirit as the roadmap in the project brief:
- Swap in your real college documents and re-run the evaluation.
- Try a larger/better LLM (or an API-based one) and compare answer quality.
- Add re-ranking of retrieved chunks, or hybrid keyword + semantic search.
- Fill in `answer_correct` / `citation_correct` in `eval_results.csv` and report real accuracy numbers.
